# Tome C2, Chapter 6: Flows and Matchings (Python companion)

Exercises first, solutions at the end. We implement Edmonds–Karp, Dinic, Hopcroft–Karp, Edmonds' blossom algorithm, minimum-cost flows by successive shortest paths and the Hungarian algorithm, and check them all against `networkx`, `scipy` and brute force. On the way we watch the bounds of the chapter at work, count Ford–Fulkerson's wasted steps, and see the threshold at which random bipartite graphs acquire a perfect matching.

*Before each solution, predict what the output will look like.*

In [ ]:
import random, math, time, itertools, heapq
from collections import deque
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
from scipy.optimize import linear_sum_assignment
random.seed(6)

## Exercises

**P6.1 (★★) Maximum flow.** Implement Edmonds–Karp (Theorem 6.2) and Dinic (Theorem 6.3) on a residual-graph data structure; compare with `networkx` on random networks; count augmentations and phases against the bounds $nm$ and $n-1$. Run a bad Ford–Fulkerson on Figure 6.2.

**P6.2 (★★) Hopcroft–Karp.** Implement Theorem 6.6, compare with `networkx`, and count phases against $2\sqrt n$. Extract Kőnig's vertex cover (Exercise 6.5) and check it.

**P6.3 (★★★) Blossoms.** Implement Edmonds' algorithm for general graphs, compare with brute force and with `networkx`, and verify the Tutte–Berge bound of Exercise 6.7 on the result.

**P6.4 (★★) Costs.** Implement successive shortest paths with potentials (Theorem 6.10) and the $O(n^3)$ assignment algorithm (Corollary 6.11); compare with `scipy.optimize.linear_sum_assignment` and `networkx.min_cost_flow`.

**P6.5 (★) Random bipartite graphs.** Each of the $n^2$ possible edges between $X$ and $Y$ is present with probability $p=c\ln n/n$. Estimate the probability of a perfect matching as a function of $c$.

## Solutions

### P6.1 Maximum flow
Residual arcs are stored in pairs: arc `e` and its reverse `e ^ 1`, so that pushing flow on one returns capacity to the other. Opposite arcs of the network are then harmless.

In [ ]:
class Network:
    def __init__(self, n):
        self.n = n; self.head = [[] for _ in range(n)]; self.to = []; self.cap = []
    def add(self, u, v, c):
        self.head[u].append(len(self.to)); self.to.append(v); self.cap.append(c)
        self.head[v].append(len(self.to)); self.to.append(u); self.cap.append(0)

def edmonds_karp(N, s, t):
    cap = N.cap[:]; flow = 0; aug = 0
    while True:
        par = [-1] * N.n; par[s] = -2; Q = deque([s])
        while Q and par[t] == -1:
            u = Q.popleft()
            for e in N.head[u]:
                if cap[e] > 0 and par[N.to[e]] == -1: par[N.to[e]] = e; Q.append(N.to[e])
        if par[t] == -1: return flow, aug
        b, v = math.inf, t
        while v != s: e = par[v]; b = min(b, cap[e]); v = N.to[e ^ 1]
        v = t
        while v != s: e = par[v]; cap[e] -= b; cap[e ^ 1] += b; v = N.to[e ^ 1]
        flow += b; aug += 1

def dinic(N, s, t):
    cap = N.cap[:]; flow = 0; phases = 0
    while True:
        level = [-1] * N.n; level[s] = 0; Q = deque([s])
        while Q:
            u = Q.popleft()
            for e in N.head[u]:
                if cap[e] > 0 and level[N.to[e]] < 0: level[N.to[e]] = level[u] + 1; Q.append(N.to[e])
        if level[t] < 0: return flow, phases
        phases += 1; ptr = [0] * N.n
        def push(u, b):                         # advance from u with at most b units; iterative would avoid deep recursion
            if u == t: return b
            while ptr[u] < len(N.head[u]):
                e = N.head[u][ptr[u]]; v = N.to[e]
                if cap[e] > 0 and level[v] == level[u] + 1:
                    got = push(v, min(b, cap[e]))
                    if got: cap[e] -= got; cap[e ^ 1] += got; return got
                ptr[u] += 1                      # the arc is saturated or leads to a dead end: retreat
            return 0
        while True:
            got = push(s, math.inf)
            if not got: break
            flow += got

def random_network(n, m, U, seed):
    rng = random.Random(seed); N = Network(n); G = nx.DiGraph(); G.add_nodes_from(range(n)); arcs = 0
    while arcs < m:
        u, v = rng.randrange(n), rng.randrange(n)
        if u == v or G.has_edge(u, v): continue
        c = rng.randint(1, U); N.add(u, v, c); G.add_edge(u, v, capacity=c); arcs += 1
    return N, G

worst_ratio = 0; max_phases = 0
for trial in range(200):
    n = random.randrange(2, 40); m = random.randrange(1, min(4 * n, n * (n - 1)) + 1)
    N, G = random_network(n, m, 20, trial)
    ref = nx.maximum_flow_value(G, 0, n - 1)
    (f1, aug), (f2, ph) = edmonds_karp(N, 0, n - 1), dinic(N, 0, n - 1)
    assert f1 == f2 == ref and aug <= n * m and ph <= n - 1
    worst_ratio = max(worst_ratio, aug / (n * m)); max_phases = max(max_phases, ph / (n - 1))
print(f"Edmonds-Karp and Dinic agree with networkx on 200 networks; augmentations used at most {worst_ratio:.1%} of nm, phases at most {max_phases:.0%} of n-1")

In [ ]:
# Figure 6.2 with an adversary that alternates through the middle arc
def bad_ford_fulkerson(C):
    cap = {('s','a'): C, ('s','b'): C, ('a','t'): C, ('b','t'): C, ('a','b'): 1}
    f = {k: 0 for k in cap}; steps = 0
    while True:
        if f[('a','b')] == 0 and f[('s','a')] < C and f[('b','t')] < C:      # s a b t
            for k in (('s','a'), ('a','b'), ('b','t')): f[k] += 1
        elif f[('a','b')] == 1 and f[('s','b')] < C and f[('a','t')] < C:    # s b a t, ab backwards
            f[('s','b')] += 1; f[('a','b')] -= 1; f[('a','t')] += 1
        else: break
        steps += 1
    return f[('s','a')] + f[('s','b')], steps

for C in (10, 100, 1000):
    N = Network(4); idx = {'s': 0, 'a': 1, 'b': 2, 't': 3}
    for (u, v), c in {('s','a'): C, ('s','b'): C, ('a','t'): C, ('b','t'): C, ('a','b'): 1}.items(): N.add(idx[u], idx[v], c)
    print(f"C={C}: bad choices reach value {bad_ford_fulkerson(C)[0]} after {bad_ford_fulkerson(C)[1]} augmentations; Edmonds-Karp after {edmonds_karp(N, 0, 3)[1]}")

In [ ]:
ns = [200, 400, 800, 1600]; tek, tdi = [], []
for n in ns:
    N, G = random_network(n, 5 * n, 1000, n)
    t0 = time.perf_counter(); a = edmonds_karp(N, 0, n - 1); tek.append(time.perf_counter() - t0)
    t0 = time.perf_counter(); b = dinic(N, 0, n - 1); tdi.append(time.perf_counter() - t0)
    print(f"n={n:5d}, m=5n: flow {a[0]:6d}; Edmonds-Karp {a[1]:4d} augmentations, {tek[-1]:5.2f} s;  Dinic {b[1]:2d} phases, {tdi[-1]:5.2f} s")

On random networks both algorithms are far below their worst-case bounds: a handful of phases and a few dozen augmentations, against $n-1$ and $nm$. Worst cases exist (Exercise 6.1 and its relatives), but they are rare in nature.

### P6.2 Hopcroft–Karp

In [ ]:
def hopcroft_karp(nx_, ny, adj):
    # adj[x] = list of y; returns matchX, matchY, number of phases
    INF = math.inf; mx = [-1] * nx_; my = [-1] * ny; phases = 0
    while True:
        dist = [INF] * nx_; Q = deque()
        for x in range(nx_):
            if mx[x] < 0: dist[x] = 0; Q.append(x)
        found = INF
        while Q:
            x = Q.popleft()
            if dist[x] >= found: continue
            for y in adj[x]:
                if my[y] < 0: found = min(found, dist[x] + 1)
                elif dist[my[y]] == INF: dist[my[y]] = dist[x] + 1; Q.append(my[y])
        if found == INF: return mx, my, phases
        phases += 1
        def dfs(x):
            for y in adj[x]:
                if (my[y] < 0 and dist[x] + 1 == found) or (my[y] >= 0 and dist[my[y]] == dist[x] + 1 and dfs(my[y])):
                    mx[x], my[y] = y, x; return True
            dist[x] = INF                      # delete the dead vertex
            return False
        for x in range(nx_):
            if mx[x] < 0: dfs(x)

def konig_cover(nx_, ny, adj, mx, my):
    # Z = vertices reachable from free X vertices by alternating paths
    ZX = [mx[x] < 0 for x in range(nx_)]; ZY = [False] * ny; Q = deque(x for x in range(nx_) if ZX[x])
    while Q:
        x = Q.popleft()
        for y in adj[x]:
            if not ZY[y] and mx[x] != y:
                ZY[y] = True
                if my[y] >= 0 and not ZX[my[y]]: ZX[my[y]] = True; Q.append(my[y])
    return [('x', x) for x in range(nx_) if not ZX[x]] + [('y', y) for y in range(ny) if ZY[y]]

worst = 0
for trial in range(200):
    rng = random.Random(trial); a, b = rng.randrange(1, 60), rng.randrange(1, 60)
    adj = [[y for y in range(b) if rng.random() < rng.choice([0.02, 0.05, 0.1])] for _ in range(a)]
    mx, my, ph = hopcroft_karp(a, b, adj)
    G = nx.Graph(); G.add_nodes_from(('x', x) for x in range(a)); G.add_nodes_from(('y', y) for y in range(b))
    G.add_edges_from((('x', x), ('y', y)) for x in range(a) for y in adj[x])
    size = sum(v >= 0 for v in mx)
    assert size == len(nx.bipartite.maximum_matching(G, top_nodes=[('x', x) for x in range(a)])) // 2
    assert all(mx[x] < 0 or (mx[x] in adj[x] and my[mx[x]] == x) for x in range(a))
    K = set(konig_cover(a, b, adj, mx, my))
    assert len(K) == size and all(('x', x) in K or ('y', y) in K for x in range(a) for y in adj[x])
    worst = max(worst, ph / (2 * math.sqrt(a + b) + 1))
print(f"Hopcroft-Karp agrees with networkx; Konig covers have the size of the matching; phases at most {worst:.0%} of 2 sqrt(n) + 1")

for n in (1000, 10000, 100000):
    rng = random.Random(n); adj = [rng.sample(range(n), 3) for _ in range(n)]
    t0 = time.perf_counter(); mx, my, ph = hopcroft_karp(n, n, adj)
    print(f"n={n:6d}, 3 random neighbours each: matching {sum(v >= 0 for v in mx)} of {n} in {ph} phases (2 sqrt(2n) = {2*math.sqrt(2*n):.0f}), {time.perf_counter()-t0:.2f} s")

### P6.3 Blossoms
This is the classical array implementation of Edmonds' algorithm: instead of building $G/B$, it keeps for every vertex the `base` of the outermost blossom containing it, which is the same as working in the contracted graph. A search from one free root grows the alternating tree by breadth-first search; `lca` finds the base of a new blossom; `mark` relabels the blossom's vertices, makes them even, and records parents so that an augmenting path through the blossom can be unwound later (this is the expansion of Lemma 6.7, done lazily). The search returns the end of the augmenting path and the parent array, from which the path is flipped.

In [ ]:
def edmonds_matching(n, adj):
    match = [-1] * n
    def search(root):
        par = [-1] * n; base = list(range(n)); even = [False] * n
        even[root] = True; Q = deque([root])
        def lca(a, b):
            seen = [False] * n
            while True:
                a = base[a]; seen[a] = True
                if match[a] < 0: break
                a = par[match[a]]
            while True:
                b = base[b]
                if seen[b]: return b
                b = par[match[b]]
        def mark(v, b, child, inblossom):
            while base[v] != b:
                inblossom[base[v]] = inblossom[base[match[v]]] = True
                par[v] = child; child = match[v]; v = par[match[v]]
        while Q:
            v = Q.popleft()
            for w in adj[v]:
                if base[v] == base[w] or match[v] == w: continue
                if w == root or (match[w] >= 0 and par[match[w]] >= 0):
                    b = lca(v, w); inblossom = [False] * n
                    mark(v, b, w, inblossom); mark(w, b, v, inblossom)
                    for i in range(n):
                        if inblossom[base[i]]:
                            base[i] = b
                            if not even[i]: even[i] = True; Q.append(i)
                elif par[w] < 0:
                    par[w] = v
                    if match[w] < 0: return w, par
                    even[match[w]] = True; Q.append(match[w])
        return -1, par
    for r in range(n):
        if match[r] < 0:
            w, par = search(r)
            while w >= 0:
                v = par[w]; nxt = match[v]
                match[w], match[v] = v, w
                w = nxt
    return match

def brute_matching(n, edges):
    best = 0
    for k in range(len(edges), 0, -1):
        if k <= best: break
        for S in itertools.combinations(edges, k):
            if len({x for e in S for x in e}) == 2 * k: return k
    return 0

def tutte_berge_bound(n, adj, A):
    G = nx.Graph(); G.add_nodes_from(set(range(n)) - set(A))
    G.add_edges_from((u, v) for u in range(n) for v in adj[u] if u not in A and v not in A)
    odd = sum(len(c) % 2 for c in nx.connected_components(G))
    return (n + len(A) - odd) // 2

for trial in range(300):
    rng = random.Random(trial); n = rng.randrange(1, 11)
    edges = [(u, v) for u in range(n) for v in range(u + 1, n) if rng.random() < 0.3]
    adj = [[] for _ in range(n)]
    for u, v in edges: adj[u].append(v); adj[v].append(u)
    match = edmonds_matching(n, adj)
    assert all(match[u] < 0 or (match[match[u]] == u and match[u] in adj[u]) for u in range(n))
    assert sum(m >= 0 for m in match) // 2 == brute_matching(n, edges)
print("Edmonds' algorithm agrees with brute force on 300 small random graphs")

for trial in range(100):
    rng = random.Random(1000 + trial); n = rng.randrange(10, 120); p = rng.choice([0.02, 0.04, 0.08])
    G = nx.gnp_random_graph(n, p, seed=trial); adj = [list(G[u]) for u in range(n)]
    size = sum(m >= 0 for m in edmonds_matching(n, adj)) // 2
    assert size == len(nx.max_weight_matching(G, maxcardinality=True))
    # Exercise 6.7: the Tutte-Berge bound holds for every set A; we test two choices
    for A in ([], [u for u in range(n) if len(adj[u]) > 3]):
        assert tutte_berge_bound(n, adj, A) >= size
print("Edmonds' algorithm agrees with networkx on 100 random graphs, and no Tutte-Berge bound is violated")

A clean experiment on why blossoms matter: run the *bipartite* alternating search (ignoring odd cycles) on general graphs and see how often it stops too early.

In [ ]:
def naive_matching(n, adj):
    # augmenting path search that never shrinks: treats a vertex as visited the first time it is reached
    match = [-1] * n
    for r in range(n):
        if match[r] >= 0: continue
        par = {r: None}; Q = deque([r]); end = None
        while Q and end is None:
            v = Q.popleft()
            for w in adj[v]:
                if w in par or w == match[v]: continue
                par[w] = v
                if match[w] < 0: end = w; break
                par[match[w]] = w; Q.append(match[w])
        while end is not None:
            v = par[end]; nxt = match[v]; match[end], match[v] = v, end; end = nxt if nxt >= 0 else None
    return sum(m >= 0 for m in match) // 2

short = 0; T = 300
for trial in range(T):
    G = nx.gnp_random_graph(30, 0.08, seed=5000 + trial); adj = [list(G[u]) for u in range(30)]
    short += naive_matching(30, adj) < sum(m >= 0 for m in edmonds_matching(30, adj)) // 2
print(f"the search without blossoms misses the maximum on {short} of {T} random graphs")

### P6.4 Costs

In [ ]:
def min_cost_flow(n, arcs, s, t, want):
    # arcs: (u, v, capacity, cost) with cost >= 0; successive shortest paths with Johnson potentials
    head = [[] for _ in range(n)]; to, cap, cost = [], [], []
    for u, v, c, k in arcs:
        head[u].append(len(to)); to.append(v); cap.append(c); cost.append(k)
        head[v].append(len(to)); to.append(u); cap.append(0); cost.append(-k)
    pot = [0] * n; flow = total = 0
    while flow < want:
        dist = [math.inf] * n; dist[s] = 0; pe = [-1] * n; H = [(0, s)]
        while H:
            d, u = heapq.heappop(H)
            if d > dist[u]: continue
            for e in head[u]:
                if cap[e] > 0:
                    nd = d + cost[e] + pot[u] - pot[to[e]]          # reduced cost, >= 0
                    if nd < dist[to[e]]: dist[to[e]] = nd; pe[to[e]] = e; heapq.heappush(H, (nd, to[e]))
        if dist[t] == math.inf: break
        for v in range(n):
            if dist[v] < math.inf: pot[v] += dist[v]
        b, v = want - flow, t
        while v != s: e = pe[v]; b = min(b, cap[e]); v = to[e ^ 1]
        v = t
        while v != s: e = pe[v]; cap[e] -= b; cap[e ^ 1] += b; total += b * cost[e]; v = to[e ^ 1]
        flow += b
    return flow, total

def hungarian(c):
    # O(n^3) assignment: successive shortest paths on the dense bipartite graph, Dijkstra with arrays
    n = len(c); u = [0.0] * (n + 1); v = [0.0] * (n + 1); p = [0] * (n + 1); way = [0] * (n + 1)
    for i in range(1, n + 1):                 # add worker i
        p[0] = i; j0 = 0; minv = [math.inf] * (n + 1); used = [False] * (n + 1)
        while True:
            used[j0] = True; i0 = p[j0]; delta = math.inf; j1 = 0
            for j in range(1, n + 1):
                if not used[j]:
                    cur = c[i0 - 1][j - 1] - u[i0] - v[j]           # reduced cost
                    if cur < minv[j]: minv[j] = cur; way[j] = j0
                    if minv[j] < delta: delta = minv[j]; j1 = j
            for j in range(n + 1):                                  # update potentials
                if used[j]: u[p[j]] += delta; v[j] -= delta
                else: minv[j] -= delta
            j0 = j1
            if p[j0] == 0: break
        while j0:                                                   # flip the alternating path
            j1 = way[j0]; p[j0] = p[j1]; j0 = j1
    sigma = [0] * n
    for j in range(1, n + 1): sigma[p[j] - 1] = j - 1
    return sigma

rng = np.random.default_rng(6)
for trial in range(200):
    n = rng.integers(1, 30); c = rng.integers(0, 100, (n, n))
    sig = hungarian(c.tolist()); r, col = linear_sum_assignment(c)
    assert sorted(sig) == list(range(n)) and sum(c[i][sig[i]] for i in range(n)) == c[r, col].sum()
    arcs = [(0, 1 + i, 1, 0) for i in range(n)] + [(1 + i, 1 + n + j, 1, int(c[i][j])) for i in range(n) for j in range(n)] + [(1 + n + j, 2 * n + 1, 1, 0) for j in range(n)]
    assert min_cost_flow(2 * n + 2, arcs, 0, 2 * n + 1, n) == (n, c[r, col].sum())
print("Hungarian and successive shortest paths agree with scipy on 200 assignment problems")

for trial in range(100):
    r_ = random.Random(trial); n = r_.randrange(3, 25); G = nx.DiGraph(); arcs = []
    for _ in range(4 * n):
        u, v = r_.randrange(n), r_.randrange(n)
        if u != v and not G.has_edge(u, v) and not G.has_edge(v, u):
            c, k = r_.randint(1, 10), r_.randint(0, 20); G.add_edge(u, v, capacity=c, weight=k); arcs.append((u, v, c, k))
    G.add_nodes_from(range(n)); F = nx.maximum_flow_value(G, 0, n - 1)
    G.nodes[0]['demand'] = -F; G.nodes[n - 1]['demand'] = F
    ref = nx.cost_of_flow(G, nx.min_cost_flow(G))
    assert min_cost_flow(n, arcs, 0, n - 1, F) == (F, ref)
print("successive shortest paths agree with networkx min_cost_flow on 100 networks")

for n in (100, 200, 400):
    c = rng.random((n, n))
    t0 = time.perf_counter(); sig = hungarian(c.tolist()); t = time.perf_counter() - t0
    print(f"n={n}: optimal assignment cost {sum(c[i][sig[i]] for i in range(n)):.4f} in {t:.2f} s")

For uniform random costs the optimal assignment cost tends to $\pi^2/6\approx1.645$ as $n\to\infty$: Mézard and Parisi predicted it in 1987 with the replica method of statistical physics, and Aldous proved it in 2001. Single samples fluctuate, but the values above are already in its neighbourhood.

### P6.5 Random bipartite graphs
Erdős and Rényi (1964) showed that the threshold for a perfect matching is the threshold for having no isolated vertex: $p=\ln n/n$. Below it, some vertex is alone; above it, almost surely a perfect matching exists.

In [ ]:
cs = np.linspace(0.4, 2.0, 17)
plt.figure(figsize=(6, 3.3))
for n, T in ((50, 300), (500, 150), (3000, 60)):
    probs = []
    for c in cs:
        p = c * math.log(n) / n; ok = 0
        for rep in range(T):
            r_ = np.random.default_rng(int(1e6 * c) + 1000 * n + rep)
            deg = r_.binomial(n, p, size=n)
            adj = [list(r_.choice(n, size=d, replace=False)) for d in deg]
            mx, _, _ = hopcroft_karp(n, n, adj)
            ok += all(v >= 0 for v in mx)
        probs.append(ok / T)
    plt.plot(cs, probs, 'o-', ms=3, label=f"n={n}")
plt.axvline(1, color='gray', ls='--'); plt.xlabel("c, where p = c ln n / n"); plt.ylabel("P(perfect matching)")
plt.legend(); plt.tight_layout(); plt.show()